In [ ]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, average_precision_score
from pyod.utils.data import precision_n_scores
from pyod.models.iforest import IForest
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from pyod.models.xgbod import XGBOD

In [ ]:
def evaluate_metrics(y_test, y_pred, y_proba=None, digits=3):
    res = {"Accuracy": round(accuracy_score(y_test, y_pred), digits),
           "Precision": round(precision_score(y_test, y_pred), digits),
           "Recall": round(recall_score(y_test, y_pred), digits),
           "F1": round(f1_score(y_test, y_pred), digits),
           "MCC": round(matthews_corrcoef(y_test, y_pred), ndigits=digits)}
    if y_proba is not None:
        res["AUC_PR"] = round(average_precision_score(y_test, y_proba), digits)
        res["AUC_ROC"] = round(roc_auc_score(y_test, y_proba), digits)
        res["PREC_N_SCORES"] = round(precision_n_scores(y_test, y_proba), digits)
    return res


def set_seed_numpy(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
features = [
    "mean", "var", "std", "len", "duration", "len_weighted", "gaps_squared", "n_peaks",
    "smooth10_n_peaks", "smooth20_n_peaks", "var_div_duration", "var_div_len",
    "diff_peaks", "diff2_peaks", "diff_var", "diff2_var", "kurtosis", "skew",
]
SEED = 2137

In [ ]:
df = pd.read_csv("data/dataset.csv", index_col="segment")

#df.plot()

X_train, y_train = df.loc[df.train==1, features], df.loc[df.train==1, "anomaly"]
X_test, y_test = df.loc[df.train==0, features], df.loc[df.train==0, "anomaly"]
X_train_nominal = df.loc[(df.anomaly==0)&(df.train==1), features]

#standardize the data to have a mean of 0 and a standar deviation of 1
#useful when values are in different units, varying length, and sampling frequency
prep = StandardScaler()
X_train_nominal2 = prep.fit_transform(X_train_nominal)
X_train2 = prep.transform(X_train)
X_test2 = prep.transform(X_test)

#print(X_test)
print(X_test2)


In [ ]:
set_seed_numpy(SEED)

In [ ]:
# supervised example

In [ ]:
import shap

In [ ]:
model = AdaBoostClassifier(random_state=SEED)
model.fit(X_train2, y_train)

y_predicted = model.predict(X_test2)
y_predicted_score = model.decision_function(X_test2)

print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))

In [ ]:
#X_test2_df = pd.DataFrame(X_test2, columns=features)
#X_explain = X_test2_df.sample(n=50, random_state=SEED)

X_explain = X_test2[:200]

explainer = shap.KernelExplainer(
    model.decision_function, shap.sample(X_train2, 500, random_state=SEED)
)

shap_values = explainer(X_explain)


In [ ]:
shap.initjs()

In [ ]:
shap.summary_plot(shap_values, X_explain)
shap.waterfall_plot(shap_values[0])
#shap.plots.bar(shap_values)
#shap.summary_plot(shap_values, X_test, plot_type="bar")
#shap.dependence_plot("var", shap_values, X_test)
#shap.summary_plot(shap_values, X_test2, plot_type="dot")

shap.plots.beeswarm(shap_values)
#shap.plots.waterfall(shap_values[7])
shap.plots.force(shap_values[0])
shap.plots.scatter(shap_values[0], color=shap_values)



In [ ]:
# unsupervised example

In [ ]:
#model = IForest(random_state=SEED)
#model.fit(X_train2)

#y_predicted = model.predict(X_test2)
#y_predicted_score = model.decision_function(X_test2)

#print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))

In [ ]:
model = IForest(random_state=SEED, contamination=.2)
model.fit(X_train2)

y_predicted = model.predict(X_test2)
y_predicted_score = model.decision_function(X_test2)

print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))

In [ ]:
import shap

In [ ]:
explainer = shap.Explainer(model)
shap_values = explainer(X_test2)

shap.summary_plot(shap_values, X_test2)
shap.waterfall_plot(shap_values[0])
#shap.plots.bar(shap_values)

shap.plots.force(shap_values)



In [ ]:
#model = RandomForestClassifier(random_state=SEED)
#model.fit(X_train2,y_train)

#y_predicted = model.predict(X_test2)
#y_predicted_score = model.predict_proba(X_test2)[:, 1]

#print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))
#y_predicted


In [ ]:
#explainer = shap.TreeExplainer(model)
#shap_values = explainer.shap_values(X_test2, check_additivity=False)


#shap.summary_plot(shap_values, X_test2)
#shap.plots.bar(shap_values)

In [ ]:
#model = XGBOD(random_state=SEED, contamination=.2)
#model.fit(X_train2, y_train)

#y_predicted = model.predict(X_test2)
#y_predicted_score = model.decision_function(X_test2)

#print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))

In [ ]:
#X_explain = X_test2[:50]

#explainer = shap.Explainer(
#    model.decision_function,
#    shap.sample(X_train2, 100, random_state=SEED)
#)

#shap_values = explainer(X_explain)


In [ ]:
#np.shape(shap_values)
#shap.summary_plot(shap_values, X_explain)
#shap.waterfall_plot(shap_values[0])
#shap.plots.bar(shap_values)

#shap.plots.force(shap_values)